# RAG

This notebook tests the RAG chatbot functionality including:
1. Document loading and chunking
2. Vector store initialization
3. Document retrieval
4. Full RAG chat with combined documentation and database info

In [ ]:
import os
import sys

sys.path.insert(0, r'c:\Users\Arjun\Desktop\Rippling\interneers-lab\backend\python')
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'django_app.settings')
import django
django.setup()
from django_app.domain.rag_service import (
    load_documents,
    chunk_documents,
    initialize_vector_store,
    retrieve_relevant_chunks,
    rag_chat,
    test_retrieval,
    get_product_info_from_db
)

print("✓ Setup complete!")

W0318 04:22:55.452000 17060 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Arjun\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✓ Setup complete!


## Test 1: Document Loading

Verify that all documentation files (Product Manual, Return Policy, Vendor FAQs) are loaded correctly.

In [2]:
documents = load_documents()

print(f"Loaded {len(documents)} documents:")
for i, doc in enumerate(documents, 1):
    print(f"  {i}. {doc.metadata['type']} ({doc.metadata['source']}) - {len(doc.page_content)} chars")

Loaded 3 documents:
  1. Product Manual (product_manual.txt) - 3144 chars
  2. Return Policy (return_policy.txt) - 4709 chars
  3. Vendor FAQs (vendor_faqs.txt) - 13450 chars


## Test 2: Document Chunking

Test the chunking strategy to ensure documents are split appropriately for retrieval.

In [3]:
chunk_sizes = [500, 1000]

for chunk_size in chunk_sizes:
    chunks = chunk_documents(documents, chunk_size=chunk_size, chunk_overlap=50)
    print(f"\nChunk size {chunk_size}:")
    print(f"  Total chunks: {len(chunks)}")
    print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
    if chunks:
        print(f"  Sample chunk: {chunks[0].page_content[:100]}...")


Chunk size 500:
  Total chunks: 60
  Average chunk size: 353 chars
  Sample chunk: PRODUCT MANUAL - Inventory Management System

Version: 2....

Chunk size 1000:
  Total chunks: 26
  Average chunk size: 817 chars
  Sample chunk: PRODUCT MANUAL - Inventory Management System

Version: 2....


## Test 3: Vector Store Initialization

Initialize the ChromaDB vector store and verify embeddings are created.

In [4]:
vector_store = initialize_vector_store(persist_dir="./test_chroma_db")
collection = vector_store._collection
print(f"✓ Vector store initialized")
print(f"  Collection name: {collection.name}")
print(f"  Document count: {collection.count()}")

✓ Vector store initialized
  Collection name: langchain
  Document count: 180


## Test 4: Document Retrieval

Test retrieving relevant chunks for different queries. This is the core RAG retrieval functionality.

In [5]:
test_queries = [
    "What's the return policy for damaged items?",
    "How do I track my order?",
    "What is the warranty period?",
    "How do I contact vendor support?",
    "Tell me about electronics quality assurance"
]

print("Retrieval Test Results:")
print("=" * 80)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = retrieve_relevant_chunks(query, top_k=3)
    for i, doc in enumerate(results, 1):
        print(f"  {i}. [{doc.metadata['type']}] {doc.page_content[:150]}...")

Retrieval Test Results:

Query: 'What's the return policy for damaged items?'
  1. [Return Policy] 2. CATEGORY-SPECIFIC RETURN POLICIES
------------------------------------
2.1 Electronics:
- 15-day return window
- Must include all original accessor...
  2. [Return Policy] 2. CATEGORY-SPECIFIC RETURN POLICIES
------------------------------------
2.1 Electronics:
- 15-day return window
- Must include all original accessor...
  3. [Return Policy] 2. CATEGORY-SPECIFIC RETURN POLICIES
------------------------------------
2.1 Electronics:
- 15-day return window
- Must include all original accessor...

Query: 'How do I track my order?'
  1. [Return Policy] 5. RETURN PROCESS
----------------
5.1 Initiating Return:
1. Log into your account and go to order history
2. Select the item to return and provide re...
  2. [Return Policy] 5. RETURN PROCESS
----------------
5.1 Initiating Return:
1. Log into your account and go to order history
2. Select the item to return and provide re...
  3. [Return

## Test 5: Database Retrieval

Test the database integration that provides current product and stock information.

In [6]:
print("Database Information:")
db_info = get_product_info_from_db()
print(f"  Total Products: {db_info.get('total_products')}")
print(f"  Categories: {', '.join(db_info.get('categories', []))}")
print(f"  Low Stock Items: {db_info.get('low_stock', [])}")

Database Information:
  Total Products: 5
  Categories: Electronics Components
  Low Stock Items: []


## Test 6: Full RAG Chat

Test the complete RAG chat that combines documentation retrieval with database information.

In [7]:
chat_test_queries = [
    "What is the return policy for damaged items?",
    "How many products do we have in inventory?",
    "Tell me about warranty periods",
    "What items are low on stock?"
]

print("Full RAG Chat Tests:")
print("=" * 80)

for query in chat_test_queries:
    print(f"\n{'='*80}")
    print(f"User: {query}")
    response = rag_chat(query, chat_history=[])
    print(f"\nAssistant: {response}")

Full RAG Chat Tests:

User: What is the return policy for damaged items?

Assistant: For damaged items:

*   **Electronics:** Defective electronics can be exchanged within 30 days of purchase.
*   **Other Categories (e.g., Clothing & Apparel):** The provided documentation does not specify a separate return policy for items explicitly labeled as "damaged." General return policies would apply, which for clothing requires items to be unworn with original tags attached within a 30-day window.

User: How many products do we have in inventory?

Assistant: I apologize, but I do not have access to the live database information regarding the total number of products currently in inventory. The "Live Database Information" section provided is empty.

To answer your question, I would need real-time data on the current stock levels across all products.

User: Tell me about warranty periods

Assistant: The standard warranty periods for our products are as follows:

*   **Electronics:** 1-year manufa

## Test 7: Chat History

Test that chat history is maintained and used for context in conversations.

In [8]:
chat_history = []

conversation = [
    "What is the return policy?",
    "And what about warranty for electronics?",
    "How do I contact support if I have issues?"
]

print("Multi-turn Conversation Test:")
print("=" * 80)

for message in conversation:
    print(f"\nUser: {message}")
    response = rag_chat(message, chat_history=chat_history)
    print(f"Assistant: {response[:200]}...")
    
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": response})

print(f"\nChat history has {len(chat_history)} messages")

Multi-turn Conversation Test:

User: What is the return policy?
Assistant: Our return policy varies by product category. Here are the details:

*   **Electronics:**
    *   You have a 15-day return window from the date of purchase.
    *   All original accessories and packag...

User: And what about warranty for electronics?
Assistant: For electronics, we offer a **1-year manufacturer warranty** from the date of purchase....

User: How do I contact support if I have issues?
Assistant: I apologize, but I encountered an error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, pleas...

Chat history has 6 messages


## Test 8: API Endpoint Test

Test the RAG API endpoints (requires Django server running).

In [1]:
import requests
try:
    response = requests.post(
        'http://localhost:8000/rag/retrieve-test/',
        json={'query': 'return policy damaged items'},
        timeout=10
    )
    if response.status_code == 200:
        data = response.json()
        print("✓ Retrieve API working")
        print(f"  Found {len(data.get('results', []))} relevant chunks")
    else:
        print(f"✗ API error: {response.status_code}")
except Exception as e:
    print(f"⚠ Could not connect to API: {e}")
    print("  (Make sure Django server is running on port 8000)")

✓ Retrieve API working
  Found 3 relevant chunks


In [2]:
try:
    response = requests.post(
        'http://localhost:8000/rag/chat/',
        json={
            'message': 'What is the return policy for damaged items?',
            'chat_history': []
        },
        timeout=30
    )
    if response.status_code == 200:
        data = response.json()
        print("✓ Chat API working")
        print(f"  Response: {data.get('response', '')[:200]}...")
    else:
        print(f"✗ API error: {response.status_code}")
        print(f"  {response.text}")
except Exception as e:
    print(f"⚠ Could not connect to API: {e}")
    print("  (Make sure Django server is running on port 8000)")

✓ Chat API working
  Response: The return policy for damaged items depends on the nature of the damage:

*   **Customer Damage:** If the product is damaged by the customer, no refund will be issued, and the item will be returned to...
